<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/PPG%E8%A8%8A%E8%99%9F%E9%87%8F%E6%B8%AC%E8%88%87%E5%88%86%E6%9E%90_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **PPG訊號量測實作單元(二)：你的脈搏與心臟似乎不同步？**


> 在這次的實驗中，我們將利用單元(一)的PPG脈波抵達瞬間，搭配ECG訊號的波峰分析，求出脈波抵達時間(PAT)！

> 根據學術文獻，PPG訊號頻率能量集中在0.01到10Hz、ECG訊號頻率能量集中在0.5到40Hz。

# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
#@title
import pandas as pd         #引入pandas函式庫，命名為np
import numpy as np         #引入numpy函式庫，命名為np
import plotly.graph_objects as go #引入plotly.graph_objects函式庫，命名為go
from google.colab import files #從google colab函式庫中，引入files函式

#強大的濾波器，不需要再轉換到頻域即可快速濾除雜訊，還能達到一樣的效果！
from scipy.signal import butter, filtfilt #從scipy.signal函式庫中，引入濾波器函式butter, filtfilt
#訊號處理常用的濾波器，可直接根據定義的頻域能量範圍，將正確的訊號過濾出來
#使用方法需要提供(時域訊號、資料取樣頻率、特定保留頻率起點、特定保留頻率終點)
def super_filter(data, frequency, save_frequency_start, save_frequency_end):   #super_filter(時域訊號, 週期, 特定保留頻率起點, 特定保留頻率終點)
  b, a = butter(3, [save_frequency_start, save_frequency_end], fs=frequency, btype='band')
  y = filtfilt(b, a, data)
  return y

# 1.將範例PPG訊號和範例ECG訊號丟進程式庫吧！

In [ ]:
#先將資料夾中的「範例PPG訊號」和「範例ECG訊號」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2. 先試著分別畫出原始與濾波後的PPG訊號吧！

In [ ]:
df_PPG = pd.read_csv('範例PPG訊號.txt') #以pandas的read_csv即可讀取檔案
data_PPG = np.array(df_PPG)              #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2_PPG = data_PPG[:,0]         #指定data中的第一行資料，建立陣列data2
#稍微設定一下畫布資訊！
fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2_PPG,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="範例PPG訊號 (原始訊號)",        #幫這張圖形物件命名
)
fig.show()                      #顯示圖形

period = 1/1000     #宣告資料取樣週期參數
frequency = 1000     #宣告資料取樣頻率參數
data3_PPG = super_filter(data2_PPG, frequency, 0.01, 10)

fig2 = go.Figure()      #建立一個圖形物件
fig2.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data3_PPG,        #新增一條線條，將取得的data3畫出
))
fig2.update_layout(                  #更新圖形的說明
    title="範例PPG訊號 (濾波後訊號)",        #幫這張圖形物件命名
)
fig2.show()                     #顯示圖形

# 3. 試著將PPG訊號轉換成So and chan的斜率訊號，並找出PPG訊號脈波抵達瞬間的位置吧！

> So and chan的斜率公式(n) = -2*X(n-2) - X(n-1) + X(n+1) + 2*X(n+2)

> PPG訊號轉換成斜率訊號，進一步找波峰，即可取得脈波抵達瞬間位置！

In [ ]:
#自行練習
#練習一：將PPG訊號轉換成So and chan的斜率訊號
#練習二：試著找到斜率訊號的波峰，取得脈波抵達瞬間位置


#-----------------------練習一提示：for迴圈 或 陣列運算-----------------------


#-----------------------練習二提示：find_peak函式-----------------------


In [ ]:
#畫出PPG訊號的斜率變化與波峰偵測位置
fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data_so_and_chan1,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='範例PPG訊號的斜率變化'    #幫此線條命名
))
fig2.add_trace(go.Scatter(           #新增一條線條在此圖形
    x=peak_list_x_PPG,              #指定其x軸的位置
    y=peak_list_y_PPG,              #指定其y軸的位置
    mode='markers',             #定義此線段不連線，僅畫出有標記的位置
    marker=dict(
        color='red',           #幫此標記以紅色標記
    ),
    name='分析之波峰位置'          #幫此線條命名
))
fig2.update_layout(                  #更新圖形的說明
    title="範例PPG訊號的斜率變化與偵測波峰", #幫這張圖形物件命名
)
fig2.show()                 #顯示圖形

# 4. 再試著分別畫出原始與濾波後的ECG訊號吧！

In [ ]:
df_ECG = pd.read_csv('範例心電訊號.txt') #以pandas的read_csv即可讀取檔案
data_ECG = np.array(df_ECG)              #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2_ECG = data_ECG[:,0]         #指定data中的第一行資料，建立陣列data2
#稍微設定一下畫布資訊！
fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2_ECG,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="範例ECG訊號 (原始訊號)",        #幫這張圖形物件命名
)
fig.show()                      #顯示圖形

period = 1/1000     #宣告資料取樣週期參數
frequency = 1000     #宣告資料取樣頻率參數
data3_ECG = super_filter(data2_ECG, frequency, 0.5, 40)

fig2 = go.Figure()      #建立一個圖形物件
fig2.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data3_ECG,        #新增一條線條，將取得的data3畫出
))
fig2.update_layout(                  #更新圖形的說明
    title="範例ECG訊號 (濾波後訊號)",        #幫這張圖形物件命名
)
fig2.show()                     #顯示圖形

# 5. 利用找波峰函式，將「ECG訊號」的波峰位置也給找出來吧！



> 找到ECG訊號的波峰位置，以利後續結合脈波抵達瞬間的位置，算出脈波抵達時間(PAT)！

In [ ]:
from scipy.signal import find_peaks #從scipy.signal函式庫中，引入找波峰函式find_peaks

#Height參數設定：請先觀察R波特化的資料秀出的波形，Height應該設定為多少才合適？

#Distance參數設定：
#1.若資料錄製時設定Fast，資料記錄頻率為1000Hz，資料記錄週期為0.001秒
#2.假設人體每次心跳最快每分鐘也不可能超過210(次/分鐘)的話，換算為秒數單位的話就是3.5(次/秒)
#3.每個收縮波峰之間間距最低秒數，就可以直接取3.5的倒數，近似為0.28秒
#4.已知資料記錄週期為0.001秒，即可得知資料點最低限值為0.28/0.001=280

height_ECG=15    #宣告波峰認定資料最低限值
distance_ECG=280  #宣告,波峰之間間隔資料點最低限值
peak_list_x_ECG = find_peaks(data3_ECG, height=height_ECG, distance=distance_ECG)[0]  #指定其x軸的位置為波峰偵測位置
peak_list_y_ECG = [data3_ECG[j] for j in peak_list_x_ECG]              #指定其y軸的位置為波峰偵測位置時的data3_ECG對應數值

print("印出波峰偵測位置的第一個點：",peak_list_x_ECG[0])
print("印出波峰偵測位置第一點的data3_ECG對應數值：",peak_list_y_ECG[0])

In [ ]:
#畫出濾波後的ECG訊號與波峰偵測位置
fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data3_ECG,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='範例ECG訊號 (濾波後訊號)'    #幫此線條命名
))
fig2.add_trace(go.Scatter(           #新增一條線條在此圖形
    x=peak_list_x_ECG,              #指定其x軸的位置
    y=peak_list_y_ECG,              #指定其y軸的位置
    mode='markers',             #定義此線段不連線，僅畫出有標記的位置
    marker=dict(
        color='red',           #幫此標記以紅色標記
    ),
    name='分析之波峰位置'          #幫此線條命名
))
fig2.update_layout(                  #更新圖形的說明
    title="範例ECG訊號 (濾波後訊號)與偵測波峰", #幫這張圖形物件命名
)
fig2.show()                 #顯示圖形

# 6. 確認ECG訊號的波峰資料與和PPG訊號的脈波抵達瞬間資料

1.   計算PAT時，每一個ECG訊號波峰應該搭配每一個在它後面，隨之而來的PPG脈波抵達瞬間。但有兩種例外情況會使紀錄的資料有誤。

2.   例外情況一：按下「開始紀錄」的瞬間，第一個ECG訊號波峰已經過去沒紀錄到，但PPG訊號的脈波又剛好到來！那就會出現第一個「脈波抵達瞬間」其實是要搭配上一個ECG訊號波峰，此情形應該剔除這一個「脈波抵達瞬間」。

3. 例外情況二：按下「停止紀錄」的瞬間，第一個ECG訊號波峰已經紀錄，但PPG訊號的脈波卻來不及到來！那就會出現最後一個「ECG訊號波峰」沒辦法與任一個PPG脈波抵達瞬間搭配，此情形應該剔除這一個「ECG訊號波峰」。

4. 當然這也能夠透過更複雜的演算法克服，例如比對資料長度或是比對兩種資料數值的大小，不過這邊就讓初學的我們用肉眼以觀察法來學習吧！

In [ ]:
#畫出濾波後的ECG訊號與PPG訊號脈波抵達瞬間觀察位置
fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data3_ECG,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='範例ECG訊號 (濾波後訊號)'    #幫此線條命名
))
fig2.add_trace(go.Scatter(           #新增一條線條在此圖形
    y=data_so_and_chan1,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='範例PPG訊號脈波抵達瞬間'    #幫此線條命名
))
fig2.update_layout(                  #更新圖形的說明
    title="範例ECG與PPG訊號脈波抵達瞬間 疊圖分析", #幫這張圖形物件命名
)
fig2.show()                 #顯示圖形

In [ ]:
#以用肉眼觀察法確認ECG訊號和PPG訊號脈波抵達瞬間的詳細資料

print('ECG訊號波峰的位置是在',peak_list_x_ECG)
print('PPG訊號脈波抵達瞬間的位置是在',peak_list_x_PPG)

print("\n")

print('ECG訊號的波峰個數是',np.size(peak_list_x_ECG))
print('PPG訊號脈波抵達瞬間個數是',np.size(peak_list_x_PPG))

In [ ]:
#如果存在例外狀況，可利用陣列分割技巧，將陣列的資料去頭或去尾
x1 =  [1,2,3,4,5]   #建立一個範例陣列
x1_cut_first = x1[1:] #利用陣列分割技巧，取x1的陣列，並從資料位置1，取到結束
x1_cut_tail = x1[:-1] #利用陣列分割技巧，取x1的陣列，並從資料位置起始，取到資料位置結束

print('範例陣列：',x1)
print('範例陣列去頭後：',x1_cut_first)
print('範例陣列去尾後：',x1_cut_tail)

# 7. 算出各個時間段的脈波抵達時間(PAT)

In [ ]:
#自行練習
#若確認資料正確，即可將ECG波峰位置減去對應的PPG脈波抵達瞬間，算出各個時間段的脈波抵達時間(PAT)





# 小結：
---
1.   ECG波峰位置搭配脈波抵達瞬間位置，對應的資料相減即為脈波抵達時間(PAT)。
2.   波峰的位置資料可能因為紀錄的起始點或結束點不佳，形成例外情形。